In [2]:
from dotenv import load_dotenv
from langgraph_runtime_inmem.checkpoint import InMemorySaver
from typing_extensions import runtime

load_dotenv()

True

# Example

- We create ColourContext class, and we store the information about colours we like and dislike in it.
- We pass this information during the time of creation of agent
- But when we check the response, the AI agent does not know our favourite colour.

In [3]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "white"
    least_favourite_colour: str = "green"

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=ColourContext
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='68269c20-517e-454e-a9cb-e4974ec14cbd'),
              AIMessage(content='I don’t know your favorite color yet. If you’d like, I can guess. My guess: blue, since it’s a very common favorite.\n\nWant me to refine the guess? Answer a couple quick questions:\n- Do you prefer cool tones (blue/green/purple) or warm tones (red/orange/yellow)?\n- Do you like bright, bold colors or soft pastels?\n- Is there a color you definitely don’t like?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 933, 'prompt_tokens': 12, 'total_tokens': 945, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 832, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id

### Why AI Agent can't tell what is our favourite colour?

- This is because context is not passed to the model directly.

![](https://i.imgur.com/gHT6Lat.png)

- Rather context is passed to the agent in an object called tool runtime, which contains the information that model has access to.

![](https://i.imgur.com/OWiGWk0.png)

- We only want to give our model the exact amount of information it needs so that we do not overload its context window.
- We create a tool call for our agent to be able to access that information

## Accessing Context

In [7]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime):
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime):
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [8]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

E:\code\langchain-basics\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...avourite_colour='green'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='c65846f0-433b-4698-9e75-b789bf8bc221'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 149, 'prompt_tokens': 149, 'total_tokens': 298, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D7QeNzZlXTdIP0C1uaILeI5AOGAxH', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c43be-e7f8-7be2-902a-544949687115-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_V77B1E8VUn4L2Q2i6ZrBXtGv', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="violet")
)

pprint(response)

E:\code\langchain-basics\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...avourite_colour='green'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='32dad085-ee5e-4a0f-bb93-501ce1b2c086'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 213, 'prompt_tokens': 149, 'total_tokens': 362, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D7QeWPhi8NciUADoOSnpLUhp2b4JR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c43bf-0ce5-7a82-a400-d5fdde769c6a-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_l6TLqIwlRDVtUiClCMMdgdYr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_

In [11]:
pprint(response["messages"][-1].content)

('Your favourite colour is violet. Would you like me to remember that for this '
 'chat, or would you like some color pairings or meanings associated with '
 'violet?')


## State

- Above we saw how to build the context_schema and how we pass it to the agent at runtime. This context is immutable, meaning our agent cannot actually update or change it itself.
- E.g. Say preferred language or role being internal is something agent learns on its own at some point in the conversation and agent wants to update its memory with that info. We can do this using agent state.
- State keeps track of history of messages, if we want to keep track of any other fields, we just add those values to our custom state.
- E.g. A custom state that keeps track of a user's favourite colour. Opposed to context, we cannot include default values in here. First we get the agent to update the state on its own.
- To do so, agent would need to access the state via the tool runtime. We create a tool that updates the colour using command function.

In [22]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

In [48]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage, HumanMessage


memory = InMemorySaver()

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state"""
    return Command(
        update={
            "favourite_colour": favourite_colour,
            "messages": [ToolMessage(
                "Successfully updated favourite colour",
                tool_call_id=runtime.tool_call_id
            )]
        }
    )

@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state"""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"


In [49]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=memory,
    state_schema=CustomState
)

In [51]:
# Step 1: Save purple to thread "1"
print("=" * 50)
print("STEP 1: Saving purple")
print("=" * 50)
response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is purple")]},
    {"configurable": {"thread_id": "1"}}
)
print(f"State after saving purple: {response['favourite_colour']}")
print(f"AI: {response['messages'][-1].content}\n")

STEP 1: Saving purple
State after saving purple: purple
AI: Got it—purple is saved as your favourite colour. Would you like me to tailor future messages to reflect that (e.g., use a purple-themed style or accents), or set any other preferences (tone, length, topics, etc.)?



In [67]:
len(response['messages'])

response['messages'][7]

AIMessage(content='Got it—purple is saved as your favourite colour. Would you like me to tailor future messages to reflect that (e.g., use a purple-themed style or accents), or set any other preferences (tone, length, topics, etc.)?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 377, 'prompt_tokens': 277, 'total_tokens': 654, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D7REhicgDwcXm922FPq9GNL61lGT3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c43e1-42cf-7562-94e9-c879aa1690e3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 277, 'output_tokens': 377, 'total_tokens': 654, 'input_token_details': {'audio'

In [52]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'purple',
 'messages': [HumanMessage(content='My favourite colour is purple', additional_kwargs={}, response_metadata={}, id='c469f086-e3dd-47f0-a0b3-92c5b1c92768'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 475, 'prompt_tokens': 158, 'total_tokens': 633, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D7RDHYKiWZsHDxG21Ktuh61mE3XOW', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c43df-ed20-7832-aace-0090f3f24017-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'purple'}, 'id': 'call_iXKou4F4zQQc1W7yKHOgr6Pk', 'type': 'tool_call'}], inval

In [68]:
# Step 2: Update to black in thread "1"
print("=" * 50)
print("STEP 2: Updating to black")
print("=" * 50)
response = agent.invoke(
    {"messages": [HumanMessage(content="Actually, my favourite colour is black")]},
    {"configurable": {"thread_id": "1"}}
)
print(f"State after updating to black: {response['favourite_colour']}")
print(f"AI: {response['messages'][-1].content}\n")

STEP 2: Updating to black
State after updating to black: black
AI: Done—the favourite colour is now black.

Would you like me to tailor future messages to reflect a black-theme style (sleek, high-contrast) or adjust other preferences (tone, length, topics, etc.)? If you have specific ideas for how you want the style to feel, tell me and I’ll apply it.



In [69]:
# Step 3: Read the colour from thread "1"
print("=" * 50)
print("STEP 3: Reading favourite colour")
print("=" * 50)
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)
print(f"State when reading: {response['favourite_colour']}")
print(f"AI: {response['messages'][-1].content}\n")

STEP 3: Reading favourite colour
State when reading: black
AI: Your favourite colour is black.

Would you like me to tailor future messages to a black-theme style (sleek, high-contrast) or adjust other preferences (tone, length, topics, etc.)?



In [70]:
# Step 4: Verify it persists in a new message
print("=" * 50)
print("STEP 4: Asking again to confirm persistence")
print("=" * 50)
response = agent.invoke(
    {"messages": [HumanMessage(content="Tell me my favourite colour again")]},
    {"configurable": {"thread_id": "1"}}
)
print(f"State: {response['favourite_colour']}")
print(f"AI: {response['messages'][-1].content}\n")

STEP 4: Asking again to confirm persistence
State: black
AI: Your favourite colour is black.

Would you like me to tailor future messages to a black-theme style (sleek, high-contrast) or adjust other preferences (tone, length, topics)?



In [ ]:
# EOF